In [1]:
# Setup: imports and basic configuration for Keras Applications inference

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.utils import load_img, img_to_array

In [2]:
# Load 3 pre-trained Keras Applications models (ImageNet weights)

from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0

models = {
    "ResNet50": ResNet50(weights="imagenet"),
    "MobileNetV2": MobileNetV2(weights="imagenet"),
    "EfficientNetB0": EfficientNetB0(weights="imagenet"),
}

list(models.keys())


['ResNet50', 'MobileNetV2', 'EfficientNetB0']

In [3]:
# Helper: preprocess + predict + decode top-3 classes for each model

from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre, decode_predictions as resnet_decode
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre, decode_predictions as mob_decode
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre, decode_predictions as eff_decode

preprocess_map = {
    "ResNet50": (resnet_pre, resnet_decode, (224, 224)),
    "MobileNetV2": (mob_pre, mob_decode, (224, 224)),
    "EfficientNetB0": (eff_pre, eff_decode, (224, 224)),
}

def predict_top3(model_name: str, img_path: str):
    pre_fn, decode_fn, target_size = preprocess_map[model_name]
    img = load_img(img_path, target_size=target_size)
    x = img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = pre_fn(x)
    preds = models[model_name].predict(x, verbose=0)
    top3 = decode_fn(preds, top=3)[0]
    return [(cls, label, float(prob)) for (cls, label, prob) in top3]


In [4]:
# Run predictions on all images in Task4-1/images and build a results table

import os
from pathlib import Path
import pandas as pd

# --- Robustly locate the 'images' folder ---
candidates = [
    Path("images"),  # if current working directory is correct
    Path.cwd() / "images",
    Path(r"E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-1_Applications_Compare\images"),
    Path(r"E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-1_Applications_Compar\images"),
]

images_dir = None
for p in candidates:
    if p.exists() and p.is_dir():
        images_dir = p
        break

if images_dir is None:
    raise FileNotFoundError(
        "Could not find the 'images' folder.\n"
        "Make sure you have an 'images' folder in the same folder as this notebook, OR inside:\n"
        "E:\\JYU\\Deep-Learning for Cognitive Computing for Developers\\Task 4\\Task4-1_Applications_Compare\\images"
    )

print("✅ Using images folder:", images_dir.resolve())

img_files = sorted([f for f in os.listdir(images_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])

if len(img_files) == 0:
    raise FileNotFoundError("No .jpg/.jpeg/.png files found inside the images folder.")

rows = []
for f in img_files:
    img_path = os.path.join(images_dir, f)
    for model_name in models.keys():
        top3 = predict_top3(model_name, img_path)
        rows.append({
            "image_file": f,
            "model": model_name,
            "top1_label": top3[0][1],
            "top1_prob": round(top3[0][2], 4),
            "top2_label": top3[1][1],
            "top2_prob": round(top3[1][2], 4),
            "top3_label": top3[2][1],
            "top3_prob": round(top3[2][2], 4),
        })

df_results = pd.DataFrame(rows)
df_results

✅ Using images folder: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-1_Applications_Compare\images


,image_file,model,top1_label,top1_prob,top2_label,top2_prob,top3_label,top3_prob
0,Apple.jpg,ResNet50,Granny_Smith,0.9312,strawberry,0.0216,hip,0.0073
1,Apple.jpg,MobileNetV2,Granny_Smith,0.2413,pomegranate,0.1432,strawberry,0.0482
2,Apple.jpg,EfficientNetB0,Granny_Smith,0.4233,pomegranate,0.0772,strawberry,0.0482
3,Rocks.jpg,ResNet50,abacus,0.3937,coil,0.2395,bagel,0.0794
4,Rocks.jpg,MobileNetV2,screw,0.1144,wool,0.0694,mortar,0.0376
5,Rocks.jpg,EfficientNetB0,mortar,0.1273,coil,0.0991,matchstick,0.0804
6,Strawberry.jpg,ResNet50,strawberry,0.9996,pineapple,0.0003,cock,0.0000
7,Strawberry.jpg,MobileNetV2,strawberry,0.9948,pineapple,0.0005,banana,0.0002
8,Strawberry.jpg,EfficientNetB0,strawberry,0.8697,pineapple,0.0032,matchstick,0.0013
9,blurry baby.jpg,ResNet50,horizontal_bar,0.8292,parallel_bars,0.0693,jean,0.0275


In [5]:
# Summary table (clean labels)

pivot_top1 = df_results.pivot_table(
    index="image_file",
    columns="model",
    values=["top1_label", "top1_prob"],
    aggfunc="first"
)

# Make labels presentation-friendly (e.g., Granny_Smith -> Granny Smith)
pivot_top1 = pivot_top1.copy()
pivot_top1[("top1_label", "EfficientNetB0")] = pivot_top1[("top1_label", "EfficientNetB0")].str.replace("_", " ", regex=False)
pivot_top1[("top1_label", "MobileNetV2")]  = pivot_top1[("top1_label", "MobileNetV2")].str.replace("_", " ", regex=False)
pivot_top1[("top1_label", "ResNet50")]     = pivot_top1[("top1_label", "ResNet50")].str.replace("_", " ", regex=False)

pivot_top1

top1_label                \
model                                 EfficientNetB0   MobileNetV2   
image_file                                                           
Apple.jpg                               Granny Smith  Granny Smith   
Rocks.jpg                                     mortar         screw   
Strawberry.jpg                            strawberry    strawberry   
blurry baby.jpg                                swing     ice lolly   
tom-and-jerry-cartoon-free-vector.jpg     wall clock    hair slide   
yes.jpg                                    ice lolly     ice lolly   

                                                           top1_prob  \
model                                        ResNet50 EfficientNetB0   
image_file                                                             
Apple.jpg                                Granny Smith         0.4233   
Rocks.jpg                                      abacus         0.1273   
Strawberry.jpg                             strawberry         0.8697   
blurry baby.jpg                        horizontal bar         0.2680   
tom-and-jerry-cartoon-free-vector.jpg      pencil box         0.0916   
yes.jpg                                        bucket         0.2352   

                                                            
model                                 MobileNetV2 ResNet50  
image_file                                                  
Apple.jpg                                  0.2413   0.9312  
Rocks.jpg                                  0.1144   0.3937  
Strawberry.jpg                             0.9948   0.9996  
blurry baby.jpg                            0.1203   0.8292  
tom-and-jerry-cartoon-free-vector.jpg      0.0748   0.3489  
yes.jpg                                    0.3673   0.2468